# Full testbed

In [9]:

import logging
import sys
from pathlib import Path

from anb.agents import ANBAgent
from anb.model import ModelSpec
from anb.networks import ErdosRenyiNetwork
from anb.testbed import PressureSchedule, Testbed, TestbedConfig

## Set config

In [ ]:
N_AGENTS = 100
M = 10
TAU = 1
LINK_PROB = 10 / N_AGENTS
BETA = 3.0
RHO = 1 / 3

# The sweep from ANBmodel.py's __main__ block, one dict per setting:
#   static low w | static high w | adaptive until external pressure | fully adaptive (each without, then with mu)
SETTINGS = [
    {"init_w": 0.2, "eps": 0.0, "mu": 0.0, "fixedBNat100": False},
    {"init_w": 0.8, "eps": 0.0, "mu": 0.0, "fixedBNat100": False},
    {"init_w": 0.2, "eps": 1.0, "mu": 0.0, "fixedBNat100": True},
    {"init_w": 0.2, "eps": 1.0, "mu": 0.0, "fixedBNat100": False},
]
# for mu in [0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0]:
#     SETTINGS.append({"init_w": 0.2, "eps": 1.0, "mu": mu, "fixedBNat100": False})
    
PRESSURES = [0, 1, 2, 4, 8, 16]
SEEDS = list(range(0, 100))  

# Legacy-named CSVs in ../simOut, where the analysis scripts look (ANBmodel.py writes to ../simOut_old_model)
OUT_DIR = Path("../simOut")
# True resumes a crashed sweep by skipping seeds whose CSV exists. Those CSVs are stale after a model change,
# so clear or switch OUT_DIR then.
SKIP_EXISTING = False
# True saves every time step for every agent (legacy detail mode, files in OUT_DIR/detailed), False the compact ensemble rows.
# Detailed CSVs are large (and legacy only used seeds=[1]), so shrink SEEDS and PRESSURES when switching this on.
DETAIL = False


In [13]:
# Timestamped INFO log, to the notebook output and to sweep.log next to the CSVs
OUT_DIR.mkdir(parents=True, exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(OUT_DIR / "sweep.log")],
    force=True,
)
log = logging.getLogger("testbed_notebook")


def make_spec(init_w, eps, mu, fixedBNat100):
    # fixedBNat100 is learning_until = 101 in the package (see docu/validation.md, deviation 2)
    agent = ANBAgent(beta=BETA, eps=eps, mu=mu, init_w=init_w, tau=TAU, n_beliefs=M,
                     learning_until=101 if fixedBNat100 else None)
    return ModelSpec(agent=agent, network_cls=ErdosRenyiNetwork,
                     network_kwargs={"link_prob": LINK_PROB}, n_agents=N_AGENTS, rho=RHO)


# One ensemble per (setting, pressure), like legacy's param_combis × seeds. Each seed's CSV is written as soon as it
# finishes, so a crash keeps everything done so far. n_jobs=-2 = all cores but one.
ensembles = [(setting, pressure) for setting in SETTINGS for pressure in PRESSURES]
for i, (setting, pressure) in enumerate(ensembles, start=1):
    log.info("ensemble %d/%d: %s, pressure=%s", i, len(ensembles), setting, pressure)
    config = TestbedConfig(PressureSchedule.single_event(strength=pressure), detail=DETAIL)
    Testbed(make_spec(**setting), config).save_ensemble(SEEDS, OUT_DIR, n_jobs=-2, skip_existing=SKIP_EXISTING)

2026-10-23 18:19:11 INFO testbed_notebook: ensemble 1/6: {'init_w': 0.2, 'eps': 1.0, 'mu': 0.0, 'fixedBNat100': False}, pressure=0
2026-10-23 18:19:11 INFO anb.testbed: ensemble start: link_prob=0.10 beta=3.00 eps=1.00 mu=0.000 lam=0.00 init_w=0.20 rho=0.33 fixedBNat100=False ext_strength=0 n_events=1 | 2 seeds, n_jobs=-2, out_dir=../simOut
2026-10-23 18:19:13 INFO anb.testbed: 1/2 seeds done, elapsed 1.5s, ETA 1.5s
2026-10-23 18:19:13 INFO anb.testbed: 2/2 seeds done, elapsed 1.5s, ETA 0.0s
2026-10-23 18:19:13 INFO anb.testbed: ensemble done in 1.5s: 2 written, 0 skipped, in ../simOut
2026-10-23 18:19:13 INFO testbed_notebook: ensemble 2/6: {'init_w': 0.2, 'eps': 1.0, 'mu': 0.0, 'fixedBNat100': False}, pressure=1
2026-10-23 18:19:13 INFO anb.testbed: ensemble start: link_prob=0.10 beta=3.00 eps=1.00 mu=0.000 lam=0.00 init_w=0.20 rho=0.33 fixedBNat100=False ext_strength=1 n_events=1 | 2 seeds, n_jobs=-2, out_dir=../simOut
2026-10-23 18:19:15 INFO anb.testbed: 1/2 seeds done, elapsed 2.